# SCF Phase 2 shard 06

Sweeps: correctness. Jobs: 55. Projected: 1.9 h
(safety-factor 1.8x applied). Grids hash: `68970a975545`.
Code source: github.com/hugogobato/scf-confounding-frontier @ tag `phase2-freeze` (pinned for
reproducibility).
Pre-registration: `docs/phase2_preregistration.md` (thresholds frozen before
any data generation; deviation register D1-D7 included there).

Resume-safe: completed cells are skipped on rerun (checkpoint parquet per
cell). If the notebook approaches the Colab wall limit it finishes the
current cell and stops cleanly; rerun to continue.

In [ ]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"
!pip install -q "numpy>=2.0" "scipy>=1.14" "pandas>=2.2" "pyarrow>=16" scikit-learn

In [ ]:
!git clone --depth 1 --branch phase2-freeze \
    https://github.com/hugogobato/scf-confounding-frontier.git scf_repo
import sys, hashlib, json
sys.path.insert(0, "scf_repo/code")
# verify the pinned code matches the manifest recorded at generation time
EXPECTED = json.loads("{\"de_formulas.py\": \"5dffb441b638\", \"simulator.py\": \"ef31ca2a201b\", \"estimators.py\": \"7e27f25b2330\", \"detection.py\": \"06586fe60b9f\", \"runners.py\": \"df67486b60f5\"}")
for fname, short in EXPECTED.items():
    h = hashlib.sha256(open(f"scf_repo/code/{fname}", "rb").read()).hexdigest()[:12]
    assert h == short, f"code mismatch: {fname} ({h} != {short})"
print("code verified against generation-time hashes")

In [ ]:
import json, time, traceback
from multiprocessing import Pool
from runners import run_cell

JOBS = json.loads("[{\"config\": {\"n\": 2000, \"p\": 1000, \"r\": 5, \"l\": [0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"e1472acfe05d\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/e1472acfe05d.parquet\", \"means_path\": \"data/sim/correctness/means/e1472acfe05d.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1000, \"r\": 5, \"l\": [2.121320343559643, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"226e6149b553\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/226e6149b553.parquet\", \"means_path\": \"data/sim/correctness/means/226e6149b553.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1000, \"r\": 5, \"l\": [2.121320343559643, 2.121320343559643, 2.121320343559643, 2.121320343559643, 2.121320343559643], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"dd0d7f971d34\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/dd0d7f971d34.parquet\", \"means_path\": \"data/sim/correctness/means/dd0d7f971d34.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1000, \"r\": 25, \"l\": [2.121320343559643, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"423f5a7816cb\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/423f5a7816cb.parquet\", \"means_path\": \"data/sim/correctness/means/423f5a7816cb.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 8000, \"p\": 800, \"r\": 1, \"l\": [0.15811388300841897], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"55e61dbb2ea7\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/55e61dbb2ea7.parquet\", \"means_path\": \"data/sim/correctness/means/55e61dbb2ea7.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 8000, \"p\": 800, \"r\": 5, \"l\": [0.9486832980505138, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"2aebaa04d942\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/2aebaa04d942.parquet\", \"means_path\": \"data/sim/correctness/means/2aebaa04d942.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 8000, \"p\": 800, \"r\": 1, \"l\": [0.9486832980505138], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"1e3d0c70f60a\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/1e3d0c70f60a.parquet\", \"means_path\": \"data/sim/correctness/means/1e3d0c70f60a.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1000, \"r\": 5, \"l\": [0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], \"theta\": 0.0, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"5df8ea00a0bf\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/5df8ea00a0bf.parquet\", \"means_path\": \"data/sim/correctness/means/5df8ea00a0bf.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1000, \"r\": 5, \"l\": [2.121320343559643, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], \"theta\": 0.0, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"4704f21004b3\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/4704f21004b3.parquet\", \"means_path\": \"data/sim/correctness/means/4704f21004b3.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1000, \"r\": 5, \"l\": [0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"63512d577606\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/63512d577606.parquet\", \"means_path\": \"data/sim/correctness/means/63512d577606.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 1000, \"r\": 5, \"l\": [2.121320343559643, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"d704c983f2ec\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/d704c983f2ec.parquet\", \"means_path\": \"data/sim/correctness/means/d704c983f2ec.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 1, \"l\": [0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"62cddeed3492\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/62cddeed3492.parquet\", \"means_path\": \"data/sim/correctness/means/62cddeed3492.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 1, \"l\": [2.6832815729997477], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"4cc35ca068f7\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/4cc35ca068f7.parquet\", \"means_path\": \"data/sim/correctness/means/4cc35ca068f7.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 5, \"l\": [0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"b862e64374ff\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/b862e64374ff.parquet\", \"means_path\": \"data/sim/correctness/means/b862e64374ff.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 5, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"6a1f0821f0ba\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/6a1f0821f0ba.parquet\", \"means_path\": \"data/sim/correctness/means/6a1f0821f0ba.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 5, \"l\": [2.6832815729997477, 2.6832815729997477, 2.6832815729997477, 2.6832815729997477, 2.6832815729997477], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"4ff7757f499b\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/4ff7757f499b.parquet\", \"means_path\": \"data/sim/correctness/means/4ff7757f499b.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 400, \"r\": 25, \"l\": [2.6832815729997477, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579, 0.4472135954999579], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"2cda696a519c\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/2cda696a519c.parquet\", \"means_path\": \"data/sim/correctness/means/2cda696a519c.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 250, \"r\": 1, \"l\": [0.3535533905932738], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"a97e76475c7c\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/a97e76475c7c.parquet\", \"means_path\": \"data/sim/correctness/means/a97e76475c7c.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 250, \"r\": 1, \"l\": [2.121320343559643], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"97a57ce0783e\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/97a57ce0783e.parquet\", \"means_path\": \"data/sim/correctness/means/97a57ce0783e.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 250, \"r\": 5, \"l\": [0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"2435b6db71fa\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/2435b6db71fa.parquet\", \"means_path\": \"data/sim/correctness/means/2435b6db71fa.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 250, \"r\": 5, \"l\": [2.121320343559643, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"23456cdd8d6b\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/23456cdd8d6b.parquet\", \"means_path\": \"data/sim/correctness/means/23456cdd8d6b.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 250, \"r\": 5, \"l\": [2.121320343559643, 2.121320343559643, 2.121320343559643, 2.121320343559643, 2.121320343559643], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"336ac775aeb6\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/336ac775aeb6.parquet\", \"means_path\": \"data/sim/correctness/means/336ac775aeb6.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 250, \"r\": 25, \"l\": [2.121320343559643, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738, 0.3535533905932738], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"b6ebb87d50c2\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/b6ebb87d50c2.parquet\", \"means_path\": \"data/sim/correctness/means/b6ebb87d50c2.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 1, \"l\": [0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"6ebf3dff64e0\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/6ebf3dff64e0.parquet\", \"means_path\": \"data/sim/correctness/means/6ebf3dff64e0.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 1, \"l\": [1.3416407864998738], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"910485c35d00\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/910485c35d00.parquet\", \"means_path\": \"data/sim/correctness/means/910485c35d00.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 5, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"8498b7e148cf\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/8498b7e148cf.parquet\", \"means_path\": \"data/sim/correctness/means/8498b7e148cf.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"aa5f5b8c7087\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/aa5f5b8c7087.parquet\", \"means_path\": \"data/sim/correctness/means/aa5f5b8c7087.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 5, \"l\": [1.3416407864998738, 1.3416407864998738, 1.3416407864998738, 1.3416407864998738, 1.3416407864998738], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"587cd43db84b\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/587cd43db84b.parquet\", \"means_path\": \"data/sim/correctness/means/587cd43db84b.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 100, \"r\": 25, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"f183955c8352\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/f183955c8352.parquet\", \"means_path\": \"data/sim/correctness/means/f183955c8352.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 50, \"r\": 1, \"l\": [0.15811388300841897], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"511eca082134\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/511eca082134.parquet\", \"means_path\": \"data/sim/correctness/means/511eca082134.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 50, \"r\": 1, \"l\": [0.9486832980505138], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"45b348525fc8\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/45b348525fc8.parquet\", \"means_path\": \"data/sim/correctness/means/45b348525fc8.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 50, \"r\": 5, \"l\": [0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"369c669d0e70\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/369c669d0e70.parquet\", \"means_path\": \"data/sim/correctness/means/369c669d0e70.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 50, \"r\": 5, \"l\": [0.9486832980505138, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"a10a3401b74a\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/a10a3401b74a.parquet\", \"means_path\": \"data/sim/correctness/means/a10a3401b74a.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 50, \"r\": 5, \"l\": [0.9486832980505138, 0.9486832980505138, 0.9486832980505138, 0.9486832980505138, 0.9486832980505138], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"aa76f936b6e2\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/aa76f936b6e2.parquet\", \"means_path\": \"data/sim/correctness/means/aa76f936b6e2.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 500, \"p\": 50, \"r\": 25, \"l\": [0.9486832980505138, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"9ee708108cd1\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 600, \"raw_path\": \"data/sim/correctness/raw/9ee708108cd1.parquet\", \"means_path\": \"data/sim/correctness/means/9ee708108cd1.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 1, \"l\": [0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"04fc097446d7\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/04fc097446d7.parquet\", \"means_path\": \"data/sim/correctness/means/04fc097446d7.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 1, \"l\": [1.3416407864998738], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"0eae8ee014c6\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/0eae8ee014c6.parquet\", \"means_path\": \"data/sim/correctness/means/0eae8ee014c6.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"d7579875aa48\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/d7579875aa48.parquet\", \"means_path\": \"data/sim/correctness/means/d7579875aa48.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"72f1076da848\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/72f1076da848.parquet\", \"means_path\": \"data/sim/correctness/means/72f1076da848.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 1.3416407864998738, 1.3416407864998738, 1.3416407864998738, 1.3416407864998738], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"0681cef7b0ba\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/0681cef7b0ba.parquet\", \"means_path\": \"data/sim/correctness/means/0681cef7b0ba.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 25, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"8ed9457cbda3\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/8ed9457cbda3.parquet\", \"means_path\": \"data/sim/correctness/means/8ed9457cbda3.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 200, \"r\": 1, \"l\": [0.15811388300841897], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"ef09b607f662\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/ef09b607f662.parquet\", \"means_path\": \"data/sim/correctness/means/ef09b607f662.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 200, \"r\": 1, \"l\": [0.9486832980505138], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"09f423cbf91c\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/09f423cbf91c.parquet\", \"means_path\": \"data/sim/correctness/means/09f423cbf91c.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 200, \"r\": 5, \"l\": [0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897], \"theta\": 0.5235987755982988, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"b776e104f7fc\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/b776e104f7fc.parquet\", \"means_path\": \"data/sim/correctness/means/b776e104f7fc.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 200, \"r\": 5, \"l\": [0.9486832980505138, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"608998b92745\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/608998b92745.parquet\", \"means_path\": \"data/sim/correctness/means/608998b92745.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 200, \"r\": 5, \"l\": [0.9486832980505138, 0.9486832980505138, 0.9486832980505138, 0.9486832980505138, 0.9486832980505138], \"theta\": 0.5235987755982988, \"profile\": \"super\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"d5adcc7c1cde\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/d5adcc7c1cde.parquet\", \"means_path\": \"data/sim/correctness/means/d5adcc7c1cde.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 200, \"r\": 25, \"l\": [0.9486832980505138, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897], \"theta\": 0.5235987755982988, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"ed47735b8656\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 200, \"raw_path\": \"data/sim/correctness/raw/ed47735b8656.parquet\", \"means_path\": \"data/sim/correctness/means/ed47735b8656.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.0, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"4508beb95dd3\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/4508beb95dd3.parquet\", \"means_path\": \"data/sim/correctness/means/4508beb95dd3.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 0.0, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"373cd916d18f\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/373cd916d18f.parquet\", \"means_path\": \"data/sim/correctness/means/373cd916d18f.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"3bc5eedd9102\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/3bc5eedd9102.parquet\", \"means_path\": \"data/sim/correctness/means/3bc5eedd9102.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 400, \"r\": 5, \"l\": [1.3416407864998738, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896, 0.22360679774997896], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"f1c6ffdb910d\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/f1c6ffdb910d.parquet\", \"means_path\": \"data/sim/correctness/means/f1c6ffdb910d.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 200, \"r\": 5, \"l\": [0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897], \"theta\": 0.0, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"08646cdfe787\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/08646cdfe787.parquet\", \"means_path\": \"data/sim/correctness/means/08646cdfe787.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 200, \"r\": 5, \"l\": [0.9486832980505138, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897], \"theta\": 0.0, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"556b7bc31412\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/556b7bc31412.parquet\", \"means_path\": \"data/sim/correctness/means/556b7bc31412.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 200, \"r\": 5, \"l\": [0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897], \"theta\": 1.5707963267948966, \"profile\": \"sub\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"d858853825f7\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/d858853825f7.parquet\", \"means_path\": \"data/sim/correctness/means/d858853825f7.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}, {\"config\": {\"n\": 2000, \"p\": 200, \"r\": 5, \"l\": [0.9486832980505138, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897, 0.15811388300841897], \"theta\": 1.5707963267948966, \"profile\": \"mixed\", \"label\": \"main\", \"q_fixed\": true, \"twin_gamma0\": false}, \"config_id\": \"a22201d68305\", \"mode\": \"correctness\", \"sweep\": \"correctness\", \"reps\": 150, \"raw_path\": \"data/sim/correctness/raw/a22201d68305.parquet\", \"means_path\": \"data/sim/correctness/means/a22201d68305.npz\", \"_rank\": 0, \"_sweep\": \"correctness\"}]")

def _safe(job):
    try:
        return run_cell(job)
    except Exception as e:
        print('[FAIL]', job['config_id'], repr(e))
        traceback.print_exc()
        return job['config_id'], -1.0

t0 = time.time()
results = []
for i, job in enumerate(JOBS):
    if time.time() - t0 > 8.6 * 3600:
        print('[WALL LIMIT] stopping cleanly after', i, 'jobs')
        break
    results.append(_safe(job))
print('shard done:', results)

In [ ]:
import hashlib, json, glob, os
manifest = {'shard_id': 6, 'files': {}}
os.makedirs('data', exist_ok=True)
for f in sorted(glob.glob('data/**/*.parquet', recursive=True)) + \
         sorted(glob.glob('data/**/*.npz', recursive=True)):
    h = hashlib.sha256(open(f, 'rb').read()).hexdigest()
    manifest['files'][f] = h
with open('data/manifest.json', 'w') as fh:
    json.dump(manifest, fh, indent=1)
print(json.dumps(manifest['files'], indent=1))

In [ ]:
import shutil
archive = shutil.make_archive('scf_shard_{:02d}'.format(6), 'zip', 'data')
print('archived:', archive)
output_file = archive
try:
    from google.colab import files
    files.download(output_file)
    print('Downloaded:', output_file)
except Exception as e:
    print('(Not on Colab / download skipped):', e)